---
title : "Talk with WISKI"
execute:
    eval: false
    output: asis
    error: false
format:
    html:
        warning: false
        error: false
        code-fold: false
        css: style.css
        panel: center
---

In de documentatie laten we voorbeelden zien hoe je verschillende tools gebruikt. 

In [ ]:
# zorg dat alle variable goed staan
import json
from setup_quarto import setup_dot_env

ctx = setup_dot_env()

## 1. Parameters

Met de tool `get_parameters` kan je de parameters van WISKI opvragen.

In [ ]:
from talk_with_wiski.models import GetLocationsInput, GetParametersInput
from talk_with_wiski.tools import get_locations, get_parameters

Dit kan voor een aantal stations

In [ ]:
get_parameter_input = GetParametersInput(station_no=["ZUN0034_KKL_OW", "0300_TDB"])
get_parameter_output = await get_parameters(ctx, get_parameter_input)

get_parameter_output.model_dump()

Of voor alle stations

In [ ]:
import numpy as np

get_parameter_input = GetParametersInput()
get_parameter_output = await get_parameters(ctx, get_parameter_input)

data_dict = get_parameter_output.data
parameters = np.unique(
    [row["stationparameter_longname"].split(" [")[0].lower() for row in data_dict]
)
print(parameters)

In [ ]:
get_parameter_input = GetParametersInput(
    station_no="BRE0301_PMP1",
)
get_parameter_output = await get_parameters(ctx, get_parameter_input)

get_parameter_output.data

## 2. Locations

Met de tool `get_locations` kan je de parameters van WISKI opvragen.

In [ ]:
get_locations_input = GetLocationsInput(max_number_of_station_ids=5)

ctx = setup_dot_env()

get_locations_output = await get_locations(ctx, get_locations_input)

get_locations_output.model_dump()

### Using geocoding is also possible

This uses `from geopy.geocoders import Nominatim`, <br>
which uses the OSM API, [See the policy](https://operations.osmfoundation.org/policies/nominatim/)<br>
 -> `No heavy uses (an absolute maximum of 1 request per second).` 

In [ ]:
get_locations_input = GetLocationsInput(
    requested_geolocation=["Loon op Zand", 5.0],
    returnfields="station_no,station_id,station_name,stationparameter_name,station_latitude,station_longitude,site_name",
)
print(input)
get_locations_output = await get_locations(ctx, input)
for item in get_locations_output.data:
    print(json.dumps(item, indent=2))
print(f"Aantal records returned {len(get_locations_output.data)}")

## 3. Sites

Met de tool `get_sites` kun je de beschikbare sites van WISKI opvragen.

Hier een aantal voorbeelden.

In [ ]:
from talk_with_wiski.models import GetSitesInput
from talk_with_wiski.tools import get_sites

get_sites_input = GetSitesInput(
    returnfields="site_no,site_id,site_name",
)
get_sites_output = await get_sites(ctx, get_sites_input)
for item in get_sites_output.data:
    print(json.dumps(item, indent=2))
print(f"Aantal records returned {len(get_sites_output.data)}")

## 4. Group lists

In [ ]:
from talk_with_wiski.models import GetGroupListInput, GetGroupListOutput
from talk_with_wiski.tools import get_group_list

# Test end time before start time
get_group_list_input = GetGroupListInput(group_type="timeseries")

get_group_list_output: GetGroupListOutput = await get_group_list(
    ctx, get_group_list_input
)
get_group_list_output.model_dump()

## 5. ParametertypeList

In [ ]:
from talk_with_wiski.models import GetParametertypeListInput, GetParametertypeListOutput
from talk_with_wiski.tools import get_parametertype_list

In [ ]:
print("Test ParametertypeList: get ids for given names")
get_parametertype_list_input = GetParametertypeListInput(
    parametertype_name=["Evaporation", "WNS*"],
)
print(get_parametertype_list_input)
get_parametertype_list_output: GetParametertypeListOutput = (
    await get_parametertype_list(ctx, get_parametertype_list_input)
)

print(f"Aantal parametertype gevonden {len(get_parametertype_list_output.data)}")
get_parametertype_list_output.model_dump()

## 6. Timeseries list

Met de tool `get_timeseries_list` kan je de beschikbare timeseries ophalen voor een locatie, site en/of een parameter.

In [ ]:
from talk_with_wiski.models import GetTimeseriesListInput, GetTimeseriesListOutput
from talk_with_wiski.tools import get_timeseries_list

Hiervoor gebruiken we de eerste op gehaalde parameter en locatie combinatie: 

In [ ]:
previously_retrieved_station = get_parameter_output.data[0]["station_no"]
previously_retrieved_parameter = [
    data["parametertype_name"] for data in get_parameter_output.data
]
previously_retrieved_station, previously_retrieved_parameter

In [ ]:
get_timeseries_list_input = GetTimeseriesListInput(
    station_no=previously_retrieved_station,
    parametertype_name=previously_retrieved_parameter,
)
get_timeseries_list_output: GetTimeseriesListOutput = await get_timeseries_list(
    ctx, get_timeseries_list_input
)

get_timeseries_list_output.model_dump()

In [ ]:
get_timeseries_list_output.model_dump()["data"][0].keys()

## 7. Timeseries values

Met de tool `get_timeseries_values` kan je de timeseries data ophalen voor een timeseries_id.

In [ ]:
from talk_with_wiski.models import GetTimeseriesValuesInput, GetTimeseriesValuesOutput
from talk_with_wiski.tools import get_timeseries_values

In [ ]:
print([item["ts_id"] for item in get_timeseries_list_output.data])

In [ ]:
get_timeseries_value_input = GetTimeseriesValuesInput(
    ts_id=[item["ts_id"] for item in get_timeseries_list_output.data],
    from_date="2025-11-01",
    period="P1D",
    number_of_tsids_to_retrieve=10,
    number_of_values_per_tsid=60,
    get_as_url=False,
    # to_date="2025-11-06"
)
get_timeseries_value_output: GetTimeseriesValuesOutput = await get_timeseries_values(
    ctx, get_timeseries_value_input
)

get_timeseries_value_output.model_dump()

In [ ]:
# length = len(get_timeseries_value_output.model_dump()['timeseries_values_list'])
# print(length)

data = get_timeseries_value_output.model_dump()
print(type(data))

for item in data["timeseries_values_list"]:
    ts_id = item["ts_id"]
    station = item["station_name"]
    data_length = len(item["data"])
    print(f"TS ID: {ts_id} | Station: {station} | Number of data points: {data_length}")

In [ ]:
[item["ts_id"] for item in get_timeseries_list_output.data]

In [ ]:
get_timeseries_value_input = GetTimeseriesValuesInput(
    ts_id=[item["ts_id"] for item in get_timeseries_list_output.data],
    from_date="2023-11-01",
    to_date="2025-11-06",
    # period="P1D",   # dit mag niet icm from_date.. to_date
    # number_of_tsids_to_retrieve=10,
    # number_of_values_per_tsid=1000,
    get_as_url=True,
)

get_timeseries_value_output: str = await get_timeseries_values(
    ctx, get_timeseries_value_input
)

print(get_timeseries_value_output)

## 8. Timeseries Type

Met de tool `get_timeseries_Type` kan je de informatie ophalen of het type tijdserie.

In [ ]:
from talk_with_wiski.models import GetTimeseriesTypeInput, GetTimeseriesTypeOutput
from talk_with_wiski.tools import get_timeseries_type

We reuse the timeseries types from when we retrieved the timeseries list

In [ ]:
ts_type_id = get_timeseries_list_output.data[0]["ts_type_id"]
ts_type_id

In [ ]:
get_timeseries_type_input = GetTimeseriesTypeInput(ts_type_id=ts_type_id)
get_timeseries_type_output: GetTimeseriesTypeOutput = await get_timeseries_type(
    ctx, get_timeseries_type_input
)
get_timeseries_type_output.model_dump()

## 9. Timeseries analysis

In [ ]:
# vind een station met waterhoogte-data
partypenm = "WNS8487"  # dit is waterhoogte
get_timeseries_list_input = GetTimeseriesListInput(parametertype_name=partypenm)
get_timeseries_list_output: GetTimeseriesListOutput = await get_timeseries_list(
    ctx, get_timeseries_list_input
)

print(get_timeseries_list_output.model_dump())

unique_ts_id = {
    item.get("ts_id") for item in get_timeseries_list_output.data if item.get("ts_id") is not None
}

print(unique_ts_id)
print(len(unique_ts_id))


In [ ]:
ts_id_list1 = list(sorted(unique_ts_id))[:10]
ts_id_list1 = ts_id_list1 + ["37481042"]
print(ts_id_list1)

In [ ]:
get_timeseries_value_input = GetTimeseriesValuesInput(
    ts_id=ts_id_list1, 
    #ts_id=["37481042"], 
    from_date="2023-11-01",
    # to_date="2024-11-06",
    period="P3M",  # dit mag niet icm from_date.. to_date
    number_of_tsids_to_retrieve=20,
    number_of_values_per_tsid=1000000,
    get_as_url=False,
)

get_timeseries_value_output: str = await get_timeseries_values(
    ctx, get_timeseries_value_input
)

# print(get_timeseries_value_output)

data = get_timeseries_value_output.model_dump()
print(type(data))

for item in data["timeseries_values_list"]:
    ts_id = item["ts_id"]
    station_name = item["station_name"]
    station_no = item["station_no"]
    data_length = len(item["data"])
    print(f"TS ID: {ts_id} | station_name: {station_name} | station_no: {station_no} | Number of data points: {data_length}")

In [ ]:
import pandas as pd

records = []

for item in data["timeseries_values_list"]:
    ts_id = item["ts_id"]
    station_name = item["station_name"]
    station_no = item["station_no"]
    for timestamp, value in item["data"]:  # unpack the [timestamp, value] pairs
        records.append(
            {
                "ts_id": ts_id,
                "station_name": station_name,
                "station_no": station_no,
                "timestamp": timestamp,
                "value": value,
            }
        )

df0 = pd.DataFrame(records)
df0["timestamp"] = pd.to_datetime(df0["timestamp"])
df0 = df0.sort_values(["ts_id","timestamp"]).reset_index(drop=True)

#stat_name_list = ['Bovenstrooms','Instroomzijde']
stat_no_list = ['1198_WTH_BOV','7132_WTH_UIT']
df1 = df0[df0["station_no"].isin(stat_no_list)]
df1 = df1.sort_values(["ts_id","timestamp"]).reset_index(drop=True)

print(df1)
print(np.unique(df1.station_no))

# get_locations_input = GetLocationsInput(station_no=stat_no_list)
# ctx = setup_dot_env()
# get_locations_output = await get_locations(ctx, get_locations_input)
# get_locations_output.model_dump()

In [ ]:
import matplotlib
import platform

print(matplotlib.get_backend())

print("Matplotlib:", matplotlib.__version__)
print("Python:", platform.python_version())

# fig, ax = plt.subplots()
# ax.plot(df["timestamp"], df["value"]) #, label=station)
# plt.legend()
# plt.xlabel("Time")
# plt.ylabel("Value")
# plt.title("Timeseries by Station")
# plt.show()

In [ ]:
# import matplotlib
# matplotlib.use('Agg')  # non-interactive backend (safe for remote)
# import matplotlib.pyplot as plt
# #%matplotlib inline

# fig, ax = plt.subplots()
# ax.plot([1,2,3], [1,4,9])
# #plt.show()
# fig.savefig(r'c:/Temp/tmp.png')

In [ ]:

def timeseries_analyse(df_in,wlev_thresholds_dict) :
    
    df=df_in.copy()
    
    timeseries_analysis = {}

    # maximum
    timeseries_analysis['max']=max(df['value'])

    # tijdsduur boven een grenswaarde
    # Step 1: mark above threshold
    df.loc[:, "above"] = df["value"] > wlev_thresholds_dict[df.iloc[0].station_no]
    # Step 2: compute time differences between consecutive timestamps
    df.loc[:, "dt"] = df["timestamp"].diff().shift(-1)  # time to next row
    # Step 3: sum durations only for rows where value is above threshold
    total_time_above = df.loc[df["above"], "dt"].sum()
    timeseries_analysis['t_above_grenswaarde']=total_time_above

    return timeseries_analysis


In [ ]:
print(df1)
print(np.unique(df1.station_no))

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import sys
#import sklearn
#from sklearn.linear_model import LinearRegression

wlev_thresholds_dict = {
    '1198_WTH_BOV' : 5.3 ,
    '7127_WTH_BEN' : -0.5,
    '7132_WTH_UIT' : 0.5,
}

stat_no_list = ['1198_WTH_BOV','7132_WTH_UIT']

for stat_no in stat_no_list :
    
    df = df1[df1["station_no"].isin([stat_no])]
    df = df.sort_values(["timestamp"]).reset_index(drop=True)

    df = df.dropna()

    # do some data-analysis on these time series
    timeseries_analysis = timeseries_analyse(df,wlev_thresholds_dict)

    # Convert timestamp to numeric (seconds since start)
    df.loc[:,'time_numeric'] = (df['timestamp'] - df['timestamp'].min()).dt.total_seconds()
    X = df['time_numeric'].values/86400
    y = df['value'].values
    # : Fit linear regression
    n = len(X)

    # Calculate means
    mean_x = sum(X) / n
    mean_y = sum(y) / n

    # Calculate slope (a) and intercept (b)
    numerator = sum((x - mean_x)*(y - mean_y) for x, y in zip(X, y))
    denominator = sum((x - mean_x)**2 for x in X)

    slope = numerator / denominator
    intercept = mean_y - slope * mean_x

    print(slope, intercept)

    y_fit = [slope*xi + intercept for xi in X]
    df['trend'] = y_fit
    slope_per_day_cm = slope*100
    
    fig = px.line(df, x="timestamp", y="value", color="station_no", title=f"Timeseries h(t) \n Aantal dagen boven threshold: {timeseries_analysis['t_above_grenswaarde']}")
    threshold=wlev_thresholds_dict[stat_no]
    fig.add_hline(y=threshold, line_dash="dash", line_color="red",
            annotation_text=f"Threshold = {threshold}", annotation_position="top left")
    fig.add_trace(go.Scatter(x=df['timestamp'],y=df['trend'], line_dash="dash", line_color="green",
            #annotation_text=f"Linear trend = {slope_per_day:.2f} cm/dag", annotation_position="top right"
            name='linear trend'))
    # Add annotation for slope/intercept
    annot_index = len(df['timestamp']) // 4
    fig.add_annotation(
        x=df['timestamp'].iloc[annot_index],  # place at last timestamp
        y=df['trend'].iloc[annot_index],      # y-coordinate on trend line
        text=f"Slope: {slope_per_day_cm:.2f} cm/dag<br>Intercept: {intercept:.2f} cm",
        # showarrow=True,
        # arrowhead=2,
        # ax=-50,  # offset x of the arrow
        # ay=-50,  # offset y of the arrow
        bgcolor="white",
        bordercolor="black",
        align="right",
    )

    fig.show()

In [ ]:
print("Python version:", sys.version)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
#print("scikit-learn version:", sklearn.__version__)